In [0]:
%sql
USE CATALOG sbsrisk_dev;
USE SCHEMA bronze;

CREATE TABLE IF NOT EXISTS bronze_sbs_indice_2024 (
  _c0 STRING, _c1 STRING, _c2 STRING, _c3 STRING, _c4 STRING, _c5 STRING, _c6 STRING,
  source_file STRING,
  periodo_mes STRING
)
USING DELTA
LOCATION 'abfss://bronze@adlsbsriskdev.dfs.core.windows.net/sbsrisk_dev/bronze/bronze_sbs_indice_2024';

In [0]:
import re
from pyspark.sql.functions import lit

# Ajusta esta base al path REAL donde dejaste los excels
base_2024 = "abfss://raw@adlsbsriskdev.dfs.core.windows.net/sbs/2024/"

# Lista archivos (si tu carpeta es distinta, acá lo verás)
files = [f.path for f in dbutils.fs.ls(base_2024) if f.path.lower().endswith(".xls") or f.path.lower().endswith(".xlsx")]
print("Archivos encontrados:", len(files))
for f in files[:5]:
    print(f)

all_dfs = []

for file_path in files:
    file_name = file_path.split("/")[-1]

    # extrae mes: en2024, fe2024, etc.
    m = re.search(r"-([a-z]{2})\d{4}", file_name.lower())
    periodo = m.group(1) if m else "unknown"

    df = (
        spark.read.format("com.crealytics.spark.excel")
        .option("header", "false")
        .option("inferSchema", "false")
        .option("treatEmptyValuesAsNulls", "true")
        .option("sheetName", "INDICE")
        .load(file_path)
    )

    df = (
        df.withColumn("source_file", lit(file_name))
          .withColumn("periodo_mes", lit(periodo))
    )

    all_dfs.append(df)

# unir todo
df_final = all_dfs[0]
for d in all_dfs[1:]:
    df_final = df_final.unionByName(d)

display(df_final.limit(20))

In [0]:
(
  df_final.write.format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("sbsrisk_dev.bronze.bronze_sbs_indice_2024")
)

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM sbsrisk_dev.bronze.bronze_sbs_indice_2024;

In [0]:
dbutils.fs.ls("abfss://raw@adlsbsriskdev.dfs.core.windows.net/")

In [0]:
dbutils.fs.ls("abfss://raw@adlsbsriskdev.dfs.core.windows.net/sbs/")

In [0]:
dbutils.fs.ls("abfss://raw@adlsbsriskdev.dfs.core.windows.net/sbs/2024/")